# Retirar uma amostra dos Datasets em PT-BR para testes

O dataset de corpus `aroeira` original é muito grande, então foi necessário rodar o notebook abaixo no **Google Colab** para conseguir uma amostra de demonstração para montagem do pipeline completo.

O dataset pode ser acessado via [Hugging Face](https://huggingface.co/datasets/Itau-Unibanco/aroeira)

In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Passo 1: Instalar as bibliotecas
!pip install datasets zstandard

# Passo 2: Importar
from datasets import Dataset, load_dataset
import os

# Configuração
DATASET_NAME = "Itau-Unibanco/aroeira"
SIZE_TRAIN = 100_000  # Amostra maior para validação/treino
SIZE_DEBUG = 1_000    # Amostra minúscula para debug rápido

OUTPUT_DIR_TRAIN = "aroeira_subset_100k"
OUTPUT_DIR_DEBUG = "aroeira_subset_1k"

DESTINATION_PATH = "/content/drive/MyDrive/aroeira_datasets/"

print(f"Iniciando processamento do dataset {DATASET_NAME}...")

# --- 1. CARREGAR EM MODO STREAMING ---
# O Aroeira tem ~34.8M de linhas.
# Usamos streaming=True para não baixar tudo.
dataset_stream = load_dataset(DATASET_NAME, split="train", streaming=True)

# --- 2. EXTRAIR A MAIOR AMOSTRA NECESSÁRIA (100k) ---
print(f"Extraindo {SIZE_TRAIN} exemplos da stream...")
dataset_head = list(dataset_stream.take(SIZE_TRAIN))

# --- 3. CRIAR OS DATASETS ---
print("\nConvertendo para objeto Dataset...")
ds_train = Dataset.from_list(dataset_head)

# Cria o dataset menor selecionando apenas os primeiros 1k do dataset maior
print(f"Criando sub-amostra de debug ({SIZE_DEBUG})...")
ds_debug = ds_train.select(range(SIZE_DEBUG))

print(f"\nDataset Treino: {ds_train}")
print(f"Dataset Debug:  {ds_debug}")

# --- 4. SALVAR NO DISCO DO COLAB ---
print(f"\nSalvando '{OUTPUT_DIR_TRAIN}' no disco local...")
ds_train.save_to_disk(OUTPUT_DIR_TRAIN)

print(f"Salvando '{OUTPUT_DIR_DEBUG}' no disco local...")
ds_debug.save_to_disk(OUTPUT_DIR_DEBUG)

# --- 5. COMPACTAR E MOVER PARA O DRIVE ---
print("\nCompactando datasets...")
!zip -r aroeira_subset_100k.zip {OUTPUT_DIR_TRAIN}
!zip -r aroeira_subset_1k.zip {OUTPUT_DIR_DEBUG}

print(f"\nCopiando arquivos para: {DESTINATION_PATH}")
# Cria a pasta no drive se ela não existir
!mkdir -p "{DESTINATION_PATH}"

!cp aroeira_subset_100k.zip "{DESTINATION_PATH}"
!cp aroeira_subset_1k.zip "{DESTINATION_PATH}"

print("\n✅ Sucesso! Ambos os arquivos (100k e 1k) foram salvos no seu Drive.")

Iniciando processamento do dataset Itau-Unibanco/aroeira...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Extraindo 100000 exemplos da stream...

Convertendo para objeto Dataset...
Criando sub-amostra de debug (1000)...

Dataset Treino: Dataset({
    features: ['text', 'url', 'access_date', 'id'],
    num_rows: 100000
})
Dataset Debug:  Dataset({
    features: ['text', 'url', 'access_date', 'id'],
    num_rows: 1000
})

Salvando 'aroeira_subset_100k' no disco local...


Saving the dataset (0/1 shards):   0%|          | 0/100000 [00:00<?, ? examples/s]

Salvando 'aroeira_subset_1k' no disco local...


Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]


Compactando datasets...
  adding: aroeira_subset_100k/ (stored 0%)
  adding: aroeira_subset_100k/dataset_info.json (deflated 63%)
  adding: aroeira_subset_100k/state.json (deflated 38%)
  adding: aroeira_subset_100k/data-00000-of-00001.arrow (deflated 62%)
  adding: aroeira_subset_1k/ (stored 0%)
  adding: aroeira_subset_1k/dataset_info.json (deflated 63%)
  adding: aroeira_subset_1k/state.json (deflated 39%)
  adding: aroeira_subset_1k/data-00000-of-00001.arrow (deflated 63%)

Copiando arquivos para: /content/drive/MyDrive/UFRPE/8º Período/Mineração de Dados Educacionais/Projeto/Datasets

✅ Sucesso! Ambos os arquivos (100k e 1k) foram salvos no seu Drive.
